# L12b Lab: Adaptive Drug Dosing with Q-Learning
Chemotherapy treats cancer with drugs that kill fast-dividing cells. The drug shrinks a tumor, a mass of cancer cells, and also damages healthy tissue. This damage, called toxicity, builds up with each dose and clears slowly between doses.

A patient receives chemotherapy in treatment cycles, with one dose in each cycle. A high dose shrinks the tumor faster, but too much toxicity kills the patient. How much drug should the patient receive in each cycle?

We'll explore this question using a Markov decision process. You will write the Q-learning update, use the update to learn a dosing policy, and compare the learned policy with the optimal policy.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Model a treatment as a Markov decision process:__ The state of the patient is the size of the tumor and the level of toxicity, and each action is a drug dose. We'll build this model and show that no fixed dose cures the patient.
> * __Compute the optimal dosing policy with value iteration:__ Value iteration uses the equations of the model to compute the best dose in every patient state. We'll compare the learned policy with this optimal policy.
> * __Implement Q-learning to learn a dosing policy:__ A Q-learning agent learns from the rewards and the next states it observes, without the equations of the model. We'll write the Q-learning update, train an agent, and compare the agent's doses with the optimal doses.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths -
include(joinpath(@__DIR__, "Include.jl")); # @__DIR__ locates this notebook's setup file

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to the Julia standard libraries, we use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) for the Q-learning agent and the policy, [Plots.jl](https://docs.juliaplots.org/stable/) for the figures, and [DataFrames.jl](https://dataframes.juliadata.org/stable/) with [PrettyTables.jl](https://ronisbr.github.io/PrettyTables.jl/stable/) for the tables. See [the package documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) and the [Julia documentation](https://docs.julialang.org/en/v1/) for the functions and types used here.

Your Q-learning function goes in [src/Compute.jl](src/Compute.jl). We call [the `L12bQLearning.myqlearning(...)` function](src/Compute.jl) by its qualified name. This way, re-running the setup cell picks up your edits.
___

## Task 1: Build the dosing environment
In this task, we'll build the environment that the Q-learning agent acts on. The environment is a Markov decision process, with states, actions, a rule that gives the next state, and rewards.

The state of the patient has two variables. The tumor burden $B\in[0,1]$ is the size of the tumor, scaled so that 1 is the largest size the tumor can reach. The toxicity $Z\in[0,1]$ is the damage the drug has built up in the patient.

The actions are the doses. In each treatment cycle $t$, we choose a dose $u_{t}\in\{0,\tfrac{1}{3},\tfrac{2}{3},1\}$, where the four values mean none, low, medium, and high.

The dose changes the state through two equations:

$$
\begin{align*}
B_{t+1} &= B_{t} + \underbrace{g\,B_{t}\,(1 - B_{t})}_{\text{tumor growth}} - \underbrace{k\,u_{t}\,B_{t}}_{\text{tumor killed}}\\
Z_{t+1} &= Z_{t} + \underbrace{b\,u_{t}}_{\text{toxicity added}} - \underbrace{c\,Z_{t}}_{\text{toxicity cleared}}
\end{align*}
$$

where $g$ is the tumor growth rate, $k$ is the fraction of the tumor a full dose kills, $b$ is the toxicity a full dose adds, and $c$ is the fraction of the toxicity the body clears each cycle.

The growth term is logistic: the tumor grows fastest at medium size and stops growing as the tumor approaches its largest size.

Value iteration and Q-learning store their values in tables, which need a finite set of states. After each cycle, we clip the new values of $B$ and $Z$ to lie between 0 and 1, and we round each value to the nearest of 41 evenly spaced levels. This gives a grid of $41^{2}$ patient states.

The treatment ends when the patient enters an absorbing state. There are two kinds of absorbing states:
* __Cure__: The tumor burden falls to $B\leq B_{\text{cure}}$. The reward is $r_{\text{cure}}>0$.
* __Toxic death__: The toxicity reaches $Z\geq Z_{\text{lethal}}$. The reward is $r_{\text{dead}}<0$. A state that meets both conditions counts as a toxic death.

A cycle that ends in neither state has a negative reward, a penalty that grows with the tumor burden, the toxicity, and the dose:

$$
\begin{equation*}
r = -\left(w_{B}\,B_{t+1} + w_{Z}\,Z_{t+1} + w_{u}\,u_{t}\right)
\end{equation*}
$$

where $w_{B}$, $w_{Z}$, and $w_{u}$ are nonnegative weights.

A dose given now changes the tumor burden and the toxicity in later cycles. To choose the best dose, we need to account for the rest of the treatment.

We store the environment in [the `MyAdaptiveDosingModel` type](src/Utility.jl), which holds the grid of patient states, the doses, and the parameters. [The `build(...)` function](src/Utility.jl) takes the parameters in a `NamedTuple` and numbers the grid states $1,2,\dots,41^{2}$.

Let's build the environment and save it in the `env::MyAdaptiveDosingModel` variable:

In [ ]:
env = build(MyAdaptiveDosingModel, (
    nlevels = 41, # 41 levels on each axis: B and Z take the values 0, 0.025, ..., 1
    doses = [0.0, 1/3, 2/3, 1.0], # the doses of the four actions: none, low, medium, high
    g = 0.15, # tumor growth rate per cycle
    k = 0.35, # fraction of the tumor a full dose kills per cycle
    b = 0.50, # toxicity a full dose adds per cycle
    c = 0.35, # fraction of the toxicity cleared per cycle
    Bcure = 0.10, # the patient is cured when B ≤ 0.10
    Zlethal = 0.90, # the patient dies of toxicity when Z ≥ 0.90
    wB = 1.0, # weight of the tumor burden in the penalty
    wZ = 1.0, # weight of the toxicity in the penalty
    wu = 0.10, # weight of the dose in the penalty
    rcure = 60.0, # reward for a cure
    rdead = -60.0, # reward for a toxic death
));
nstates = env.nlevels^2; # the number of states, 41 × 41
nactions = length(env.doses); # the number of actions, one per dose

We'll follow a treatment that starts from a large tumor and no toxicity, $(B_{0}, Z_{0}) = (0.90, 0)$. [The `stateindex(...)` function](src/Utility.jl) returns the number of the grid state nearest a pair $(B, Z)$. [The `decode(...)` function](src/Utility.jl) does the reverse, and returns the pair $(B, Z)$ of a state number:

In [ ]:
s₀ = stateindex(env, 0.90, 0.0); # the starting state: a large tumor and no toxicity
decode(env, s₀) # check: (B, Z) of the starting state

### Try the four fixed doses
A policy gives the dose to use in each state. The simplest policy is a fixed dose, which uses the same dose in every state. Let's try the four fixed doses before we compute the optimal policy.

[The `simulate(...)` function](src/Utility.jl) follows a policy from a starting state until the patient is cured, the patient dies of toxicity, or 100 cycles pass. We call the third outcome a timeout.

Let's follow each fixed dose from the starting state:

In [ ]:
let
    df = DataFrame(); # one row per fixed dose
    labels = ["none", "low", "medium", "high"];
    for a ∈ 1:nactions
        π_fixed = fill(a, nstates); # the same action a in every state
        result = simulate(env, π_fixed, s₀);
        push!(df, (schedule = labels[a], dose = round(env.doses[a], digits = 2),
            outcome = String(result.outcome), cycles = result.steps));
    end
    pretty_table(df; backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

No fixed dose cures the patient. With no dose or the low dose, the tumor never shrinks to the cure level. The medium and high doses raise the toxicity to the lethal level.

To cure the patient, the dose must change with the state of the patient. We call this adaptive dosing.
___

## Task 2: Compute the optimal policy with value iteration
In this task, we compute the best dose for every patient state with value iteration. In Task 3, we'll compare the learned policy with this optimal policy.

Value iteration needs the reward and the next state of each dose in each state. [The `world(...)` function](src/Utility.jl) applies the equations of Task 1 and returns both.

Value iteration updates the value of each state with the Bellman backup. In this model, each dose leads to exactly one next state $s^{\prime}$, and the backup has no sum over next states:

$$
\begin{equation*}
U(s)\gets\max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\,U(s^{\prime})\right)
\end{equation*}
$$

where $U(s)$ is the value of state $s$, the largest total discounted reward that the treatment can collect starting from $s$. The set $\mathcal{A}$ holds the four doses, $R(s,a)$ is the reward of dose $a$ in state $s$, and $\gamma$ is the discount factor.

[The `solve(...)` function](src/Utility.jl) for the dosing environment applies this backup to every state until the values stop changing. The function returns the optimal action-value table $Q^{*}$, with one row per state and one column per action. The entry $Q^{*}(s,a)$ is the total discounted reward of giving dose $a$ in state $s$ and then following the optimal policy.

[The `policy(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.policy-Tuple{Matrix{Float64}}) picks the best action in each row of the table.

Let's compute the optimal action values and the optimal policy:

In [ ]:
γ = 0.95; # discount factor
Qstar = solve(env; γ = γ); # value iteration: the optimal action-value table
π_star = policy(Qstar); # the optimal policy: the best action in each state

Let's follow the optimal policy from the starting state and print the dose it gives each cycle:

In [ ]:
optimal = simulate(env, π_star, s₀); # follow the optimal policy from the large tumor
println("Outcome: ", optimal.outcome, " after ", optimal.steps, " cycles");
println("Doses: ", round.(optimal.dose, digits = 2));

The optimal policy cures the patient. In the early cycles, the policy gives high doses to shrink the tumor, with rest cycles at dose 0 to let the toxicity clear. The last cycles use medium and low doses.

The printed list has one more entry than the number of cycles. The last entry, 0, belongs to the state after the final cycle, when the treatment has ended.

### Visualize the optimal policy and the treatment
Let's draw the optimal policy as a heatmap. The color of each square is the optimal dose in that patient state $(B, Z)$, and the absorbing states are blank. The white line is the treatment from the starting state:

In [ ]:
let
    # the levels of the grid, 0, 0.025, ..., 1 -
    levels = range(0.0, stop = 1.0, length = env.nlevels) |> collect;

    # the optimal dose in each state; NaN (blank) in the absorbing states -
    M = fill(NaN, env.nlevels, env.nlevels); # rows are toxicity levels, columns are tumor levels
    for iB ∈ 1:env.nlevels
        for iZ ∈ 1:env.nlevels
            s = env.states[(iB, iZ)];
            if (isterminal(env, s) == false)
                M[iZ, iB] = env.doses[π_star[s]];
            end
        end
    end

    # draw the policy, the thresholds, and the treatment -
    p = heatmap(levels, levels, M, c = :viridis, colorbar_title = "optimal dose",
        xlabel = "tumor burden B", ylabel = "toxicity Z", legend = :bottomleft);
    hline!(p, [env.Zlethal], color = :red, ls = :dash, lw = 2, label = "lethal toxicity");
    vline!(p, [env.Bcure], color = :green, ls = :dash, lw = 2, label = "cure");
    plot!(p, optimal.B, optimal.Z, lw = 2, c = :white, marker = :circle, ms = 3, label = "treatment");
    p
end

The treatment starts at the bottom right and moves left toward the cure line. Each high dose pushes the toxicity up toward the lethal line, and each rest cycle lets the toxicity fall before the next dose.

Let's draw the same treatment over time:

In [ ]:
let
    cycles = range(0, stop = length(optimal.B) - 1, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
    p = plot(cycles, optimal.B, lw = 2, marker = :circle, ms = 3, label = "tumor burden B",
        xlabel = "treatment cycle", ylabel = "level or dose", legend = :outerright, size = (760, 420), margin = 5Plots.mm);
    plot!(p, cycles, optimal.Z, lw = 2, marker = :square, ms = 3, label = "toxicity Z");
    plot!(p, cycles, optimal.dose, seriestype = :steppost, lw = 2, ls = :dash, c = :purple, label = "dose u");
    hline!(p, [env.Zlethal], color = :red, ls = :dot, label = "lethal toxicity");
    hline!(p, [env.Bcure], color = :green, ls = :dot, label = "cure");
    p
end

The tumor burden falls with each high dose and grows back a little during each rest. The toxicity rises with each high dose, falls during each rest, and peaks at 0.875, just below the lethal level of 0.90.
___

## Task 3: Implement Q-learning and learn a dosing policy
In Task 2, value iteration used the equations of the model. In this task, we suppose the agent does not know those equations. The agent can only give a dose and observe the reward and the next state that [the `world(...)` function](src/Utility.jl) returns.

Q-learning uses these observations to estimate the action-value table $Q$, with one row per state and one column per action. After each dose, the agent updates one entry of the table:

$$
\begin{equation*}
Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)
\end{equation*}
$$

where $0<\alpha\leq 1$ is the learning rate and $\gamma$ is the discount factor. The term $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$ is the temporal difference target, and its difference from $Q(s,a)$ is the temporal difference error.

The agent repeats this update over many simulated treatments, which we call episodes. Here is the full algorithm:

__Initialize__: Given the state space $\mathcal{S}$, the action space $\mathcal{A}$, the absorbing states, a discount factor $0\leq\gamma<1$, a number of episodes $N\geq 1$, and a maximum number of steps per episode $d\geq 1$, set $Q(s,a)\gets 0$ and the visit count $n(s,a)\gets 0$ for all $s\in\mathcal{S}$ and $a\in\mathcal{A}$.

For $i = 1$ to $N$ __do__:
1. Set the exploration probability $\epsilon$ of episode $i$ from its schedule.
2. Choose a starting state $s$ that is not absorbing, uniformly at random, and set the step counter $t\gets 0$.
3. While $s$ is not absorbing and $t<d$ __do__:
    - Choose the action $a$: roll a random number $p\in[0,1]$.
        - If $p\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime})$ (exploitation).
    - Take action $a$, and observe the reward $r$ and the next state $s^{\prime}$ from the world.
    - Count the visit, $n(s,a)\gets n(s,a)+1$, and set the learning rate $\alpha\gets 1/n(s,a)$.
    - Update the action value with the temporal difference error:
      $$Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$$
    - Update the state $s\gets s^{\prime}$ and the step counter $t\gets t+1$.

__Extract Policy__: For each state $s\in\mathcal{S}$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a)$.

> __Training:__ The agent trains with these settings:
> * Each episode starts from a random state that is not absorbing. The episode runs until the patient is cured, the patient dies, or 80 cycles pass.
> * The learning rate of the $n$-th update of a pair $(s,a)$ is $1/n$. This rate meets the convergence conditions of the [lecture](../L12a/CHEME-5800-L12a-Lecture-QLearning-Fall-2026.ipynb).
> * The exploration probability $\epsilon$ falls in a straight line from 1, reaches 0.05 after 66.5 percent of the episodes, and then stays at 0.05.

We store the agent in a `MyQLearningAgentModel` from the course package, which holds the states, the actions, a learning rate $\alpha$, the discount factor $\gamma$, and the table $Q$. [The `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/) makes the agent.

The agent also needs the states that end a treatment, the cures and the toxic deaths. [The `isterminal(...)` function](src/Utility.jl) returns `true` for these states.

Let's build the agent, and collect the absorbing states in the `absorbing_states::Set{Int64}` set:

In [ ]:
agent = build(MyQLearningAgentModel, (
    states = collect(1:nstates), # the states 1, 2, ..., nstates
    actions = collect(1:nactions), # the actions 1, 2, 3, 4
    α = 0.1, # learning rate field (myqlearning(...) uses 1/n instead)
    γ = γ, # the same discount factor as value iteration
    Q = zeros(nstates, nactions), # the table starts at zero
));

absorbing_states = Set{Int64}(); # the states that end a treatment: cured or dead
for s ∈ 1:nstates
    if (isterminal(env, s) == true)
        push!(absorbing_states, s);
    end
end

### Implement Our Version
Open [`src/Compute.jl`](src/Compute.jl) and complete [the `myqlearning(...)` function](src/Compute.jl). A TODO comment marks each of the three lines you write:

> __Implementation tasks__
>
> * __TODO 1:__ Choose the greedy action in the exploitation branch of the epsilon-greedy rule.
> * __TODO 2:__ Compute the temporal difference target.
> * __TODO 3:__ Move $Q(s,a)$ toward the target with the learning rate $\alpha$.

Our version returns a `NamedTuple` holding the learned table `Q` and the counts `visits`, where `visits[s,a]` is the number of times the agent took action `a` in state `s`. A rerun with the same `seed` gives the same result.

Save the file, re-run the setup cell to reload your edits, then train the agent. The untouched function throws an error.

In [ ]:
result = L12bQLearning.myqlearning(agent, env; worldmodel = world, absorbing = absorbing_states,
    episodes = 200_000, maxsteps = 80, seed = 1234); # the untouched stub throws an error
π_q = policy(result.Q); # the learned policy: the best action in each row of the learned table

### Check: Your Q-Learning
We check our version on a small problem with three states and two actions:
* In state 1, action 1 moves to state 2 and earns $-1$. Action 2 stays in state 1 and earns $-2$.
* In state 2, action 1 moves to state 3 and earns $10$. Action 2 moves back to state 1 and earns $-1$.
* State 3 is absorbing.

In the checks, state 2 is also absorbing, which makes state 1 the only starting state. [The `mdpworld(...)` function](src/Utility.jl) returns the next state and the reward of each move.

We set $\epsilon=0$, which makes the agent greedy, and $\gamma=0.5$. We check three things:
* __The greedy action and the target:__ We start from $Q(1,1)=6$ and $Q(1,2)=4$. The greedy action is 1, which moves to state 2 and earns $-1$.

  The first update of a pair has $\alpha=1$. The update sets $Q(1,1)$ to the target $-1+0.5\cdot 0=-1$, because the row of the absorbing state 2 is zero.
* __The average of the targets:__ We start from $Q(1,1)=0$ and $Q(1,2)=10$, and take two steps. Both steps choose action 2, which keeps the agent in state 1 and earns $-2$.

  The first target is $-2+0.5\cdot 10=3$, and the first update sets $Q(1,2)=3$. The second target is $-2+0.5\cdot 3=-0.5$. With $\alpha=1/2$, the second update gives $Q(1,2)=3+\tfrac{1}{2}\left(-0.5-3\right)=1.25$, the average of the two targets.
* __Absorbing rows stay zero:__ A treatment ends as soon as the patient is cured or dies. In the dosing run, the agent never visits or updates the rows of these absorbing states.

Let's run the checks:

In [ ]:
@testset "My Q-learning" begin

    # the three-state problem: T[s, s′, a] and R[s, a] -
    T_small = zeros(3, 3, 2);
    R_small = zeros(3, 2);
    T_small[1, 2, 1] = 1.0; R_small[1, 1] = -1.0; # state 1, action 1: to state 2, reward -1
    T_small[1, 1, 2] = 1.0; R_small[1, 2] = -2.0; # state 1, action 2: stay in state 1, reward -2
    T_small[2, 3, 1] = 1.0; R_small[2, 1] = 10.0; # state 2, action 1: to state 3, reward 10
    T_small[2, 1, 2] = 1.0; R_small[2, 2] = -1.0; # state 2, action 2: back to state 1, reward -1
    T_small[3, 3, 1] = 1.0; T_small[3, 3, 2] = 1.0; # state 3 is absorbing: we stay put
    small = build(MyMDPProblemModel, (𝒮 = [1, 2, 3], 𝒜 = [1, 2], T = T_small, R = R_small, γ = 0.9));

    # the greedy action is 1, and the first update sets Q(1,1) to the target -1 -
    greedy_agent = build(MyQLearningAgentModel, (states = [1, 2, 3], actions = [1, 2],
        α = 0.1, γ = 0.5, Q = [6.0 4.0; 0.0 0.0; 0.0 0.0]));
    greedy_step = L12bQLearning.myqlearning(greedy_agent, small; worldmodel = mdpworld,
        absorbing = Set([2, 3]), episodes = 1, maxsteps = 1, ϵ_start = 0.0, ϵ_floor = 0.0);
    @test greedy_step.visits == [1 0; 0 0; 0 0] # one move: action 1 in state 1
    @test greedy_step.Q ≈ [-1.0 4.0; 0.0 0.0; 0.0 0.0]

    # two steps of action 2: Q(1,2) is the average of the targets 3 and -0.5 -
    average_agent = build(MyQLearningAgentModel, (states = [1, 2, 3], actions = [1, 2],
        α = 0.1, γ = 0.5, Q = [0.0 10.0; 0.0 0.0; 0.0 0.0]));
    two_steps = L12bQLearning.myqlearning(average_agent, small; worldmodel = mdpworld,
        absorbing = Set([2, 3]), episodes = 1, maxsteps = 2, ϵ_start = 0.0, ϵ_floor = 0.0);
    @test two_steps.visits == [0 2; 0 0; 0 0] # two moves: action 2 in state 1, twice
    @test two_steps.Q ≈ [0.0 1.25; 0.0 0.0; 0.0 0.0]

    # the rows of the absorbing states of the dosing run are never visited or updated -
    for s ∈ absorbing_states
        @test all(result.Q[s, :] .== 0.0)
        @test all(result.visits[s, :] .== 0)
    end
end;

### Follow the learned policy
__Prediction:__ After 200,000 episodes, will the learned policy cure the patient from the starting state: {yes, with the same doses as the optimal policy; yes, with different doses; or no}?

Let's follow the learned policy from the same starting state:

In [ ]:
learned = simulate(env, π_q, s₀); # follow the learned policy from the large tumor
println("Outcome: ", learned.outcome, " after ", learned.steps, " cycles");
println("Doses: ", round.(learned.dose, digits = 2));

The learned policy cures the patient with the same doses as the optimal policy. The agent learned these doses from the rewards and the next states it observed.

### Compare the two policies on more states
One treatment tests only the states that the treatment passes through. To test more states, we start the optimal policy from the tumor burdens $B_{0}=0.10, 0.15,\dots,0.95$ with no toxicity, and we collect the states that the optimal policy visits before the treatment ends.

In each visited state, we compare the dose of the learned policy $\pi_{q}$ with the dose of the optimal policy $\pi^{*}$. Where the two doses differ, the value lost by the learned dose is $Q^{*}(s,\pi^{*}(s)) - Q^{*}(s,\pi_{q}(s))$:

In [ ]:
let
    # the states the optimal policy visits, from starting tumor burdens 0.10, 0.15, ..., 0.95 -
    visited = Set{Int64}();
    for B₀ ∈ 0.10:0.05:0.95
        s = stateindex(env, B₀, 0.0);
        for t ∈ 1:100 # at most 100 cycles
            if (isterminal(env, s) == true)
                break; # cured or dead: the treatment has ended
            end
            push!(visited, s);
            (s, _) = world(env, s, π_star[s]); # the _ discards the reward
        end
    end

    # compare the two policies on the visited states -
    number_same = 0; # states where the two policies choose the same action
    largest_loss = 0.0; # the most value lost where they differ
    smallest_value = Inf; # the smallest optimal value of a visited state
    largest_value = -Inf; # the largest optimal value of a visited state
    for s ∈ visited
        value = Qstar[s, π_star[s]]; # the optimal value of state s
        smallest_value = min(smallest_value, value);
        largest_value = max(largest_value, value);
        if (π_q[s] == π_star[s])
            number_same = number_same + 1;
        else
            loss = value - Qstar[s, π_q[s]]; # the value lost by the learned action
            largest_loss = max(largest_loss, loss);
        end
    end

    # show -
    println("Visited states: ", length(visited));
    println("Same action: ", number_same, " (", round(100*number_same/length(visited), digits = 1), "%)");
    println("Optimal values of the visited states: ", round(smallest_value, digits = 1), " to ", round(largest_value, digits = 1));
    println("Largest value lost by a different action: ", round(largest_loss, digits = 2));
end

The two policies choose the same dose in about three quarters of the visited states. In the other states, the learned dose loses at most about 3 units of discounted reward. For scale, the optimal values of the visited states lie between about 20 and 60.

The states where the doses differ were visited less often during training, and the two best doses in those states have similar values.

Let's also follow both policies from four starting tumor burdens:

In [ ]:
let
    df = DataFrame(); # one row per starting tumor burden
    for B₀ ∈ [0.30, 0.50, 0.70, 0.90]
        s = stateindex(env, B₀, 0.0); # this start, with no toxicity
        a = simulate(env, π_star, s); # the optimal policy
        b = simulate(env, π_q, s); # the learned policy
        push!(df, (B₀ = B₀, optimal = String(a.outcome), optimal_cycles = a.steps,
            learned = String(b.outcome), learned_cycles = b.steps));
    end
    pretty_table(df; backend = :text, display_size = (-1, -1),
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

Both policies cure the patient from all four starting tumor burdens. The learned policy takes at most one more cycle than the optimal policy.
___

## Summary
In this lab, we modeled a course of chemotherapy as a Markov decision process, computed the optimal dosing policy with value iteration, and learned a dosing policy with our own Q-learning function.

> __Key Takeaways:__
>
> * __Dosing is a sequential decision problem:__ A larger dose kills more of the tumor and adds more toxicity, and both changes carry into later cycles. No fixed dose cured the patient, and the best dose depended on the state of the patient.
>
> * __The optimal dosing schedule:__ Value iteration used the equations of the model to find a schedule of high doses with rest cycles that let the toxicity clear. The schedule kept the toxicity below the lethal level while the tumor shrank to the cure level.
>
> * __Learning the doses from simulated treatments:__ Our Q-learning agent used only observed rewards and next states, and the agent gave the same doses as the optimal policy from the large starting tumor. In the states we compared, the learned dose lost little value wherever the two policies chose different doses.

We can now change the doses, the rewards, or the number of episodes and see how the optimal policy and the learned policy respond.
___